# 03 - Ingresos y distribución (EPH)

Análisis de la distribución del ingreso a partir de la EPH (INDEC), usando los parquets
compilados por **00_preparacion_bases**.

**Enfoque:** medidas de **distribución y desigualdad** (no se estima pobreza por línea/
canastas, que requeriría cargar la CBA/CBT del INDEC y deflactar). Las métricas usadas son
**unit-free** (Gini, participaciones por decil, ratios), por lo que no requieren ajustar
por inflación y son comparables a lo largo del tiempo.

**Contenido:**
1. Distribución del ingreso per cápita familiar (`IPCF`), último trimestre.
2. Participación en el ingreso por decil + ratio D10/D1.
3. Índice de Gini — serie 2017-2025.
4. Evolución de la desigualdad: ratio D10/D1 y participación del 10% más rico vs. 40% más pobre.

**Variables (base Hogar replicadas a Personas):** `IPCF` (ingreso per cápita familiar),
`PONDIH` (ponderador de ingresos familiares). Ver `.claude/memoria_EPH.md`.

## 1. Setup (Colab)

Clona el repo, monta Drive y copia los parquets a disco local (evita la desconexión del
mount). Requiere haber corrido el notebook 00.

In [ ]:
import sys, os, shutil, glob

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROCESSED = "/content/drive/MyDrive/carga_EPH/processed"
PROCESSED_DIR = "/content/processed_local"
os.makedirs(PROCESSED_DIR, exist_ok=True)
for src in glob.glob(os.path.join(DRIVE_PROCESSED, "eph_T*.parquet")):
    dst = os.path.join(PROCESSED_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
n = len(glob.glob(os.path.join(PROCESSED_DIR, "eph_T*.parquet")))
print(f"Parquets locales listos en {PROCESSED_DIR}: {n} trimestres")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_panel, list_available_quarters, _period_tag

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

quarters = list_available_quarters()
ULTIMO = quarters[-1]
print("Último trimestre:", _period_tag(*ULTIMO))

def weighted_quantile(values, q, weights):
    """Cuantil ponderado (q en [0,1])."""
    s = np.argsort(values)
    v, w = np.asarray(values)[s], np.asarray(weights)[s]
    cw = np.cumsum(w) - 0.5 * w
    cw /= np.sum(w)
    return np.interp(q, cw, v)

def weighted_gini(values, weights):
    """Índice de Gini ponderado (0 = igualdad perfecta, 1 = desigualdad máxima)."""
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    s = np.argsort(v)
    v, w = v[s], w[s]
    p = np.cumsum(w) / np.sum(w)            # share acumulado de población
    L = np.cumsum(v * w) / np.sum(v * w)    # share acumulado de ingreso (Lorenz)
    p = np.concatenate(([0], p))
    L = np.concatenate(([0], L))
    area = np.sum((L[1:] + L[:-1]) / 2 * (p[1:] - p[:-1]))  # área bajo Lorenz
    return 1 - 2 * area

## 2. Distribución del ingreso per cápita familiar (último trimestre)

Se usa `IPCF` con ponderador `PONDIH`. Se excluyen los registros sin ingreso válido
(NaN). Para visualizar se recorta el 1% superior (cola muy larga).

In [ ]:
ing = load_panel(columns=["IPCF", "PONDIH"], quarters=[ULTIMO], out_dir=PROCESSED_DIR)
ing = ing[ing["IPCF"].notna() & ing["PONDIH"].notna() & (ing["IPCF"] >= 0)].copy()

qs = [0.10, 0.25, 0.50, 0.75, 0.90]
perc = {f"P{int(q*100)}": weighted_quantile(ing["IPCF"], q, ing["PONDIH"]) for q in qs}
print(f"Percentiles del IPCF - {_period_tag(*ULTIMO)} (pesos):")
for k, v in perc.items():
    print(f"  {k}: ${v:,.0f}")

tope = weighted_quantile(ing["IPCF"], 0.99, ing["PONDIH"])
viz = ing[ing["IPCF"] <= tope]
plt.hist(viz["IPCF"], bins=60, weights=viz["PONDIH"], color="#4C72B0")
plt.axvline(perc["P50"], color="#C44E52", ls="--", label=f"Mediana ${perc['P50']:,.0f}")
plt.xlabel("IPCF ($)")
plt.ylabel("Población (ponderada)")
plt.title(f"Distribución del ingreso per cápita familiar - {_period_tag(*ULTIMO)}")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Participación en el ingreso por decil (último trimestre)

Deciles de `IPCF` calculados ponderadamente. Para cada decil, su participación en el
ingreso total. El ratio D10/D1 resume la brecha entre el decil más rico y el más pobre.

In [ ]:
def deciles_share(df):
    """Devuelve la participación (%) en el ingreso de cada decil ponderado de IPCF."""
    cortes = [weighted_quantile(df["IPCF"], q / 10, df["PONDIH"]) for q in range(1, 10)]
    dec = np.searchsorted(cortes, df["IPCF"].values, side="right") + 1  # 1..10
    df = df.assign(decil=dec)
    ing_decil = df.groupby("decil").apply(
        lambda g: np.sum(g["IPCF"] * g["PONDIH"]), include_groups=False
    )
    return ing_decil / ing_decil.sum() * 100

share = deciles_share(ing)
print("Participación en el ingreso por decil (%):")
print(share.round(1))
ratio = share.loc[10] / share.loc[1]
print(f"\nRatio D10/D1: {ratio:.1f}")

ax = share.plot(kind="bar", color="#55A868")
ax.set_xlabel("Decil de IPCF")
ax.set_ylabel("% del ingreso total")
ax.set_title(f"Participación en el ingreso por decil - {_period_tag(*ULTIMO)}")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Índice de Gini — serie 2017-2025

Gini del `IPCF` ponderado por trimestre. Va de 0 (igualdad perfecta) a 1 (desigualdad máxima).

In [ ]:
todo = load_panel(columns=["IPCF", "PONDIH"], out_dir=PROCESSED_DIR)
todo = todo[todo["IPCF"].notna() & todo["PONDIH"].notna() & (todo["IPCF"] >= 0)].copy()

def metricas(g):
    sh = deciles_share(g)
    return pd.Series({
        "gini": weighted_gini(g["IPCF"].values, g["PONDIH"].values),
        "ratio_d10_d1": sh.loc[10] / sh.loc[1],
        "share_top10": sh.loc[10],
        "share_bottom40": sh.loc[1:4].sum(),
    })

evol = todo.groupby(["ANIO", "TRIMESTRE"]).apply(metricas, include_groups=False)
evol.index = [f"{y}T{p}" for y, p in evol.index]
evol.round(3)

In [ ]:
ax = evol["gini"].plot(marker="o", ms=4, color="#C44E52")
ax.set_ylabel("Índice de Gini")
ax.set_title("Desigualdad del ingreso per cápita familiar (Gini, EPH ponderado)")
ax.set_xticks(range(len(evol.index)))
ax.set_xticklabels(evol.index, rotation=90)
plt.tight_layout()
plt.show()

## 5. Evolución de la desigualdad: ratio D10/D1 y participación top10 / bottom40

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

evol["ratio_d10_d1"].plot(ax=ax1, marker="o", ms=4, color="#8172B3")
ax1.set_ylabel("Ratio D10 / D1")
ax1.set_title("Brecha de ingresos: decil más rico / decil más pobre")

evol["share_top10"].plot(ax=ax2, marker="o", ms=4, label="10% más rico", color="#C44E52")
evol["share_bottom40"].plot(ax=ax2, marker="o", ms=4, label="40% más pobre", color="#4C72B0")
ax2.set_ylabel("% del ingreso total")
ax2.set_title("Participación en el ingreso: top 10% vs. bottom 40%")
ax2.set_xticks(range(len(evol.index)))
ax2.set_xticklabels(evol.index, rotation=90)
ax2.legend()

plt.tight_layout()
plt.show()

---
Notebook sobre los parquets de `00_preparacion_bases`. Métricas de distribución unit-free
(no requieren deflactar). Para estimar pobreza por línea habría que incorporar la CBA/CBT
del INDEC por período y región — queda como posible extensión futura.